In [66]:
import pandas as pd
import re
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize,sent_tokenize
from sklearn.model_selection import train_test_split
from nltk.stem import WordNetLemmatizer

import gensim
from gensim.models import Word2Vec, KeyedVectors
from gensim.utils import simple_preprocess

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report

In [44]:
dataset=pd.read_csv('../dataset/spam_ham.txt',sep='\t',names=['result','email'])

In [45]:
dataset

,result,email
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [46]:
lemmatizer=WordNetLemmatizer()

In [47]:
# preprocessing

corpus=[]
for i in range(len(dataset)):
    rm_sep=re.sub('[^a-zA-Z]',' ',dataset['email'][i])
    words=word_tokenize(rm_sep)
    word=[lemmatizer.lemmatize(w) for w in words if w not in set(stopwords.words('english'))]
    corpus.append(' '.join(word))



In [48]:
corpus

['Go jurong point crazy Available bugis n great world la e buffet Cine got amore wat',
 'Ok lar Joking wif u oni',
 'Free entry wkly comp win FA Cup final tkts st May Text FA receive entry question std txt rate T C apply',
 'U dun say early hor U c already say',
 'Nah I think go usf life around though',
 'FreeMsg Hey darling week word back I like fun still Tb ok XxX std chgs send rcv',
 'Even brother like speak They treat like aid patent',
 'As per request Melle Melle Oru Minnaminunginte Nurungu Vettam set callertune Callers Press copy friend Callertune',
 'WINNER As valued network customer selected receivea prize reward To claim call Claim code KL Valid hour',
 'Had mobile month U R entitled Update latest colour mobile camera Free Call The Mobile Update Co FREE',
 'I gon na home soon want talk stuff anymore tonight k I cried enough today',
 'SIX chance win CASH From pound txt CSH send Cost p day day TsandCs apply Reply HL info',
 'URGENT You week FREE membership Prize Jackpot Txt word

In [49]:
words=[]

for sent in corpus:
    sentence=sent_tokenize(sent)
    for word in sentence:
        words.append(simple_preprocess(word))

In [50]:
words

[['go',
  'jurong',
  'point',
  'crazy',
  'available',
  'bugis',
  'great',
  'world',
  'la',
  'buffet',
  'cine',
  'got',
  'amore',
  'wat'],
 ['ok', 'lar', 'joking', 'wif', 'oni'],
 ['free',
  'entry',
  'wkly',
  'comp',
  'win',
  'fa',
  'cup',
  'final',
  'tkts',
  'st',
  'may',
  'text',
  'fa',
  'receive',
  'entry',
  'question',
  'std',
  'txt',
  'rate',
  'apply'],
 ['dun', 'say', 'early', 'hor', 'already', 'say'],
 ['nah', 'think', 'go', 'usf', 'life', 'around', 'though'],
 ['freemsg',
  'hey',
  'darling',
  'week',
  'word',
  'back',
  'like',
  'fun',
  'still',
  'tb',
  'ok',
  'xxx',
  'std',
  'chgs',
  'send',
  'rcv'],
 ['even',
  'brother',
  'like',
  'speak',
  'they',
  'treat',
  'like',
  'aid',
  'patent'],
 ['as',
  'per',
  'request',
  'melle',
  'melle',
  'oru',
  'minnaminunginte',
  'nurungu',
  'vettam',
  'set',
  'callertune',
  'callers',
  'press',
  'copy',
  'friend',
  'callertune'],
 ['winner',
  'as',
  'valued',
  'network',
  

In [51]:
model=gensim.models.Word2Vec(words)

In [52]:
model.wv.similar_by_word('good')

[('well', 0.9996310472488403),
 ('hope', 0.9995689392089844),
 ('dear', 0.9995535612106323),
 ('really', 0.999531626701355),
 ('great', 0.999496340751648),
 ('day', 0.9994815587997437),
 ('much', 0.9994619488716125),
 ('love', 0.9994606375694275),
 ('amp', 0.999454140663147),
 ('night', 0.9994515180587769)]

In [53]:
def avg_word2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)  # return zero vector

    return np.mean(vectors, axis=0)

In [54]:
!pip install tqdm

In [55]:
from tqdm import tqdm

In [56]:

#apply for the entire sentences
import numpy as np
X=[]
for i in tqdm(range(len(words))):
    X.append(avg_word2vec(words[i]))

100%|██████████| 5568/5568 [00:00<00:00, 9012.65it/s] 


In [57]:
X

[array([-0.23307233,  0.13894631,  0.06069079, -0.0023329 ,  0.04040033,
        -0.40254876,  0.05570351,  0.47310942, -0.12122472, -0.09783762,
        -0.17898193, -0.30296424, -0.04057608,  0.07232309,  0.05729824,
        -0.11672444,  0.0367239 , -0.3256198 ,  0.00725213, -0.5061614 ,
         0.15542331,  0.12245694,  0.07327602, -0.09922984, -0.10818073,
         0.06554873, -0.16824964, -0.21301064, -0.18753718,  0.12054761,
         0.2859481 ,  0.13271247,  0.01923228, -0.07509268, -0.1282603 ,
         0.32876292, -0.02093824, -0.33743915, -0.22422905, -0.5121301 ,
         0.01433168, -0.26612183,  0.00984525, -0.02263709,  0.19728304,
        -0.19629525, -0.08554857, -0.03784761,  0.16343242,  0.07596588,
         0.14005376, -0.16788414, -0.05487168,  0.06102696, -0.23589115,
         0.2609243 ,  0.18775845, -0.05211365, -0.30158746,  0.09817875,
         0.05457319,  0.08896702, -0.07334055,  0.04002369, -0.32659766,
         0.19111277,  0.11753289,  0.21873282, -0.3

In [58]:
X_new=np.array(X)

In [60]:
## Dependent Features
## Output Features
y = dataset[list(map(lambda x: len(x)>0 ,corpus))]
y=pd.get_dummies(y['result'])
y=y.iloc[:,0].values

In [62]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20)

In [61]:
classifier=RandomForestClassifier()

In [63]:

classifier.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [64]:

y_pred=classifier.predict(X_test)

In [67]:
print(accuracy_score(y_test,y_pred))

0.966786355475763


In [68]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       False       0.95      0.79      0.86       149
        True       0.97      0.99      0.98       965

    accuracy                           0.97      1114
   macro avg       0.96      0.89      0.92      1114
weighted avg       0.97      0.97      0.97      1114

